# Lab 04: The Planning Pattern for Adaptive Timeline Reconstruction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab04_planning_pattern/03_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

---

### What this lab is about

In Lab 03 the AI decided its next step only after it saw the last result, one small check at a
time. **Planning** works differently. For a bigger question with several connected parts, the AI
first writes an **ordered plan** for the whole job, the way an examiner sketches an investigation
plan before touching the evidence. Then, when a new piece of evidence breaks one of the plan's
assumptions, it **revises the plan** instead of carrying on as if nothing changed.

**The question you will answer:** a phone went missing for about 30 minutes. What communication
happened on it during that window, and what can the evidence actually confirm?

**The twist:** you first plan with only part of the evidence. Then a held-back **network record**
arrives. It shows the phone was offline, so a WhatsApp image attachment inside the window cannot
be called a confirmed delivery. The plan has to change.

### How this notebook flows

- **Part 0:** set up. Install one library, connect to a free AI model, write the case files, and
  load the planning toolbox.
- **Part A:** you walk the plan, replan, report loop by hand, one AI request per cell.
- **Part B:** a ready-made `PlanningAgent` runs the same workflow, including a self-review.
- **Your answer:** you write your own timeline and compare the two runs.

Run the cells from top to bottom, in order. Each step uses what the step before it made.

The AI is a helper, not the decision maker. It can be wrong. You are the examiner: you make the
final call, and every claim must point back to a piece of evidence.

## Part 0: One-Time Setup

These four steps get the notebook ready. They install one library, connect you to a **free
AI model** hosted online by a company called Groq, write the case files, and load the planning
toolbox. Nothing is installed on your own computer.

### Step 0.1: Install the helper library

**Purpose:** the notebook needs one small add-on, the `openai` library, to send questions to the
AI and bring the answers back. Groq accepts the same format, so this library works with it.

**Input:** nothing from you. The library is downloaded into this online session only.

**How it works:** `%pip install` is a notebook command that installs a Python library. The `-q`
part keeps the install log short.

**Output:** a short install log. Warnings are normal. Step 0.2 uses the library to connect to the AI.

In [ ]:
# Install the one library this notebook needs. It goes into this online session only.
%pip install -q openai

### Step 0.2: Connect to a free AI model (Groq)

**Purpose:** connect the notebook to the AI model that will write and revise the plans in this lab.

**Input:** your free Groq **API key** (a private password that lets the notebook use the AI).
To get one:

1. Open **https://console.groq.com/keys**, sign in, and create a key.
2. Run the cell below and paste the key when asked. It stays hidden and is **not saved** into the
   notebook.

**How it works:** the cell asks for your key without showing it, opens a connection to Groq, and
picks the first AI model from a short list that your key can use. It also adds a small safety
helper that keeps each reply short enough for the free plan and waits and tries again when the
service is busy. Finally, it sends a one-word test question.

**Output:** a line that starts with `Model OK:` and shows the model's name and its one-word reply.
This step creates `client` (the connection to the AI) and `MODEL` (the name of the AI model in
use). Every AI step from Step A.2 onward uses these two. If you see a list of model names and an
error instead, follow the message: put one of the listed names first in `PREFERRED` and run the
cell again.

In [ ]:
# Connect this notebook to a free AI model hosted by Groq, and check that it answers.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: carries your messages to the AI and back (Groq uses this format)

# Ask for your key only if this session does not already have it.
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()

# The web address of Groq's service. Every question for the AI is sent here.
BASE_URL = 'https://api.groq.com/openai/v1'
# A short list of free AI models that work well for these labs, best first. Groq retires models
# from time to time, so the cell uses the first one your key can reach and prints its name.
# Edit the list to try another model.
PREFERRED = ['openai/gpt-oss-120b', 'qwen/qwen3.8-27b']

# Build the client: the connection that sends questions to the AI and brings back replies.
# max_retries=8: the free plan limits how much text you can send each minute. If you go over,
# Groq says 'try again in a few seconds', and the client waits and tries again instead of
# stopping with an error.
client = OpenAI(base_url=BASE_URL, api_key=os.environ['GROQ_API_KEY'], max_retries=8)

# Every AI request in this notebook goes through the small helper below. It does two things:
# 1. It caps the length of each reply, so you stay inside the free plan's per-minute limit.
#    Groq measures text in tokens (small pieces of words) and counts your question PLUS the
#    longest allowed reply. Replies are capped at 1,500 tokens, or 900 for Qwen, whose free plan
#    allows only 1,000 reply tokens a minute. (Any single request can still set its own cap.)
# 2. The gpt-oss model sometimes tries a built-in shortcut instead of writing plain text, and Groq
#    rejects that ('model called a tool'). The helper then asks again, up to 3 tries in total,
#    with a reminder to reply in plain text.
_create = client.chat.completions.create
def _create_within_limits(*args, **kw):
    kw.setdefault('max_tokens', 900 if 'qwen' in str(kw.get('model')) else 1500)
    for attempt in range(3):
        try:
            return _create(*args, **kw)
        except Exception as err:
            if 'called a tool' not in str(err) or attempt == 2:
                raise
            kw['messages'] = list(kw.get('messages', [])) + [
                {'role': 'user', 'content': 'Reply in plain text only; do not call functions.'}]
# From now on, every AI request made through the client uses the helper above.
client.chat.completions.create = _create_within_limits

# Ask Groq which models your key can use, then pick the first match from PREFERRED.
available = [m.id for m in client.models.list().data]
MODEL = next((m for m in PREFERRED if m in available), None)
if MODEL is None:
    print('None of PREFERRED is on your key right now. Models your key can use:')
    for m in available:
        print('  -', m)
    raise SystemExit('Put one of these chat models first in PREFERRED and re-run this cell.')

# Send one tiny test question, so you know the connection works before the case starts.
probe = client.chat.completions.create(
    model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
print('Model OK:', MODEL, '->', (probe.choices[0].message.content or '').strip())

### Step 0.3: Write the case evidence files

**Purpose:** put the case evidence on disk so the notebook can read it, the same way an examiner
works from files exported off a phone.

**Input:** the six evidence files for case `PL-2026-011`, stored inside the cell below as text.
They are synthetic (made up for teaching) and match the copies in the course repository:

- `artifact_manifest.json`: a summary of the case: the device, how it was acquired, the hash of
  the original image, and the case time window.
- `unlock_events.csv`: when the phone was unlocked and locked.
- `call_log.csv`: the one outgoing call.
- `whatsapp_events.csv`: the WhatsApp chat activity, a deleted message, and an image attachment record.
- `network_status.csv`: when the phone was online and when it was offline.
- `chain_of_custody.csv`: who handled the evidence, and when.

**How it works:** the cell makes a folder called `data/` and writes each file into it. A file that
is already there is left alone, never overwritten.

**Output:** one line per file, for example `wrote data/call_log.csv`. (The line says `wrote` even
when the file was already there from an earlier run.) Step A.1 reads all six files.

In [ ]:
# Write the six case evidence files into a folder called data/, so later steps can read them.
from pathlib import Path  # Path: a readable way to refer to files and folders by name

# ARTIFACTS is a dictionary (a list of name -> value pairs): each file name -> the text inside it.
ARTIFACTS = {'artifact_manifest.json': '{\n  "case_id": "PL-2026-011",\n  "lab": "Lab 04 - Planning Pattern",\n  "device": {\n    "platform": "Android",\n    "model": "Pixel 8",\n    "os_version": "Android 14"\n  },\n  "acquisition": {\n    "type": "full_file_system",\n    "tool": "Cellebrite UFED 7.70",\n    "acquired_at_utc": "2026-02-12T13:11:42Z",\n    "original_image_sha256": "4bc2ff573aa8bf3d6b0fb7a81341a7ef34df7b64c1d50368f6dd80e38f4a95ce"\n  },\n  "analysis_timezone": "America/New_York",\n  "incident_window_utc": {\n    "start": "2026-02-11T20:55:00Z",\n    "end": "2026-02-11T21:25:00Z"\n  },\n  "note": "Synthetic classroom dataset for planning-pattern demo"\n}\n', 'unlock_events.csv': 'timestamp_utc,event,source\n2026-02-11T20:55:03Z,device_unlocked,keyguard_log\n2026-02-11T21:25:31Z,device_locked,keyguard_log\n', 'call_log.csv': 'timestamp_utc,direction,phone_number,duration_seconds,call_type\n2026-02-11T21:08:14Z,outgoing,+1-555-0184,42,cellular_voice\n', 'whatsapp_events.csv': 'timestamp_utc,event,chat_id,details\n2026-02-11T21:12:07Z,chat_opened,client_thread_18,"Conversation opened from recent chats"\n2026-02-11T21:13:12Z,message_deleted,client_thread_18,"Deleted outbound message with image attachment placeholder"\n2026-02-11T21:13:19Z,attachment_event,client_thread_18,"Image attachment record observed in app cache metadata"\n', 'network_status.csv': 'start_utc,end_utc,network_state,notes\n2026-02-11T20:55:00Z,2026-02-11T21:10:00Z,online,"Mobile data available"\n2026-02-11T21:10:00Z,2026-02-11T21:26:59Z,offline,"No mobile data or Wi-Fi connectivity recorded"\n2026-02-11T21:27:04Z,2026-02-11T21:40:00Z,online,"Mobile data restored"\n', 'chain_of_custody.csv': 'timestamp_utc,handler,action,notes\n2026-02-12T13:11:42Z,Investigator_A,acquired_image,"UFED acquisition to encrypted evidence drive"\n2026-02-12T13:38:57Z,Investigator_A,computed_hash,"SHA-256 verified against acquisition log"\n2026-02-13T08:46:12Z,Analyst_B,created_working_copy,"Exported access, call, app, and network artifacts"\n'}

# Make the data/ folder. exist_ok=True means: no error if the folder is already there.
data_dir = Path('data'); data_dir.mkdir(exist_ok=True)
# A loop: repeat the indented lines once for each file. Only missing files are written.
for name, text in ARTIFACTS.items():
    fp = data_dir / name
    if not fp.exists():
        fp.write_text(text)
    print(('wrote ' if fp.stat().st_size else 'exists ') + str(fp))

### Step 0.4: Load the planning toolbox

**Purpose:** load the ready-made `PlanningAgent`, the packaged planning agent you will use in
Part B.

**Input:** nothing from earlier steps. All of the code is inside the cell below.

**How it works:** the cell defines the agent's standing instructions (the **prompt**: the
instructions we give the AI) and the `PlanningAgent` itself, which can do three things:

- `build_initial_plan`: write a first ordered plan for a task.
- `revise_plan`: update a plan when new evidence arrives.
- `run`: write a plan, then review it and improve it, a few times if needed.

You do not need to read every line. This copy is adapted from The Neural Maze's
agentic-patterns-course (MIT License).

**Output:** nothing is printed. Part B uses `PlanningAgent`. Part A does the same moves by hand, so
you see them clearly first.

In [ ]:
# The planning toolbox: the packaged PlanningAgent used in Part B. Run it once and move on.
# You do NOT need to read every line. Skim the names: PLANNING_SYSTEM_PROMPT (the agent's standing
# instructions) and PlanningAgent with build_initial_plan, revise_plan, and run.
# Adapted from The Neural Maze agentic-patterns-course (MIT License), (c) 2024 The Neural Maze.

# ----------------------------------------------------------------- 1. the memory (chat history)
# Each message is a small dictionary: who is speaking ('system', 'user', or 'assistant', meaning
# the AI) and what they said ('content').
def build_prompt_structure(prompt: str, role: str, tag: str = "") -> dict:
    if tag:
        prompt = f"<{tag}>{prompt}</{tag}>"
    return {"role": role, "content": prompt}


def update_chat_history(history: list, msg: str, role: str):
    history.append(build_prompt_structure(prompt=msg, role=role))


# Send a list of messages to the AI and return just the text of its reply.
def completions_create(client, messages, model) -> str:
    response = client.chat.completions.create(messages=list(messages), model=model)
    return str(response.choices[0].message.content)


# A list of messages that forgets its oldest message once it reaches a size limit
# (total_length=-1 means "no limit").
class ChatHistory(list):
    def __init__(self, messages=None, total_length=-1):
        super().__init__(messages or [])
        self.total_length = total_length

    def append(self, msg):
        if len(self) == self.total_length:
            self.pop(0)
        super().append(msg)


class FixedPrefixChatHistory(ChatHistory):
    """Keeps an initial prefix pinned (system + original task) while newer messages rotate."""

    def __init__(self, messages=None, total_length=-1, pinned_prefix_len=1):
        super().__init__(messages, total_length)
        self.pinned_prefix_len = pinned_prefix_len

    # When the memory is full, forget the oldest message AFTER the pinned ones, so the
    # instructions and the original task are never forgotten.
    def append(self, msg):
        if self.total_length != -1 and len(self) == self.total_length:
            self.pop(self.pinned_prefix_len)
        list.append(self, msg)


# ------------------------------------------------------------- 2. the agent's standing instructions
# Sent to the AI with every request the agent makes. They describe how to build and revise a plan.
PLANNING_SYSTEM_PROMPT = """
You are a digital forensics planning agent.

Your job is to break a forensic question into an ordered investigation plan, identify what evidence is needed,
and revise the plan when new observations expose a missing dependency, contradiction, or timing conflict.

When building an initial plan:
- state the investigation goal in plain language
- produce an ordered step-by-step plan
- note the evidence needed for each major step
- identify clear replanning triggers

When revising a plan:
- preserve still-useful steps
- reorder steps when new observations change dependencies
- explain how the new observations change the investigation path
- keep the revised plan concise and evidence-focused

Always avoid unsupported conclusions. Treat planning as an aid to the student, not as final decision authority.
"""


# ----------------------------------------------------------------- 3. the packaged planning agent
class PlanningAgent:
    """A planning-first agent for building and revising investigation plans.

    Entry points:
    - build_initial_plan(user_msg)            -> the first ordered plan
    - revise_plan(user_msg, plan, observations) -> a plan updated with new evidence
    - run(user_msg, max_replans)              -> plan, self-review, and improve until [[OK]]
    """

    def __init__(self, model="openai/gpt-oss-120b", system_prompt="", client=None):
        if client is None:
            raise ValueError("PlanningAgent requires a `client` (chat.completions.create).")
        self.client = client
        self.model = model
        self.system_prompt = system_prompt

    # Any extra instructions you pass in, followed by the standing planning instructions.
    def _build_system_prompt(self) -> str:
        return (self.system_prompt + "\n" + PLANNING_SYSTEM_PROMPT).strip()

    # One fresh, one-off request: the instructions plus one message. Nothing from earlier
    # requests is remembered, so everything the AI needs must be inside user_msg.
    def _complete(self, user_msg: str) -> str:
        history = [
            build_prompt_structure(self._build_system_prompt(), "system"),
            build_prompt_structure(user_msg, "user"),
        ]
        return completions_create(self.client, history, self.model)

    # Ask for a first plan, in four parts, for the task you give it.
    def build_initial_plan(self, user_msg: str) -> str:
        request = (
            "Build an initial investigation plan for the following forensic task.\n\n"
            "Return:\n1. Investigation goal\n2. Ordered steps\n3. Evidence needed for each major step\n"
            "4. Replanning triggers\n\n"
            f"Task:\n{user_msg}"
        )
        return self._complete(request)

    # Ask for an updated plan. The task, the current plan, and the new evidence all go into
    # one request, so the AI can see what to keep and what to change.
    def revise_plan(self, user_msg: str, current_plan: str, observations: str) -> str:
        request = (
            "Revise the current forensic investigation plan using the new observations below.\n\n"
            "Return:\n1. What changed\n2. Revised ordered steps\n3. Updated evidence priorities\n"
            "4. Remaining replanning triggers\n\n"
            f"Original task:\n{user_msg}\n\n"
            f"Current plan:\n{current_plan}\n\n"
            f"New observations:\n{observations}"
        )
        return self._complete(request)

    def run(self, user_msg: str, max_replans: int = 3) -> str:
        # A running memory for the review conversation. The first two messages (instructions and
        # task) always stay; older review messages drop out once it holds the maximum number.
        history = FixedPrefixChatHistory(
            [
                build_prompt_structure(self._build_system_prompt(), "system"),
                build_prompt_structure(user_msg, "user"),
            ],
            total_length=max(6, (max_replans * 2) + 2),
            pinned_prefix_len=2,
        )

        # Build the first plan (a separate one-off request), then add it to the memory.
        plan = self.build_initial_plan(user_msg)
        update_chat_history(history, plan, "assistant")

        # Self-review: ask the AI to check its own plan, up to max_replans times.
        for _ in range(max_replans):
            review_prompt = (
                "Review the current investigation plan.\n"
                "If the plan is already well ordered, evidence-focused, and includes clear replanning triggers, "
                "reply exactly [[OK]].\n"
                "Otherwise, identify the most important missing dependency, contradiction, or observation "
                "that should trigger replanning."
            )
            update_chat_history(history, review_prompt, "user")
            review = completions_create(self.client, history, self.model)
            update_chat_history(history, review, "assistant")

            # [[OK]] means the AI is satisfied with the plan, so stop early.
            if "[[OK]]" in review:
                print("Plan accepted by self-review. Stopping.")
                break

            # Otherwise, ask for an improved plan that fixes what the review found, then loop back.
            revision_request = (
                "Revise the current plan using the following review notes.\n\n"
                f"Current plan:\n{plan}\n\n"
                f"Review notes:\n{review}"
            )
            update_chat_history(history, revision_request, "user")
            plan = completions_create(self.client, history, self.model)
            update_chat_history(history, plan, "assistant")

        return plan

## The Case in One Minute

**Case `PL-2026-011`:** a caseworker's Google Pixel 8 went missing for about 30 minutes during
evening field visits. You must reconstruct what communication happened on it inside that window.

| UTC time | What the records show |
|---|---|
| 20:55:03 | device unlocked (start of the missing period) |
| 21:08:14 | outgoing call to +1-555-0184, 42 seconds |
| 21:10:00 | mobile data drops offline |
| 21:12:07 | WhatsApp chat opened |
| 21:13:12 | an outbound message with an image placeholder is deleted |
| 21:13:19 | an image attachment record appears in the app cache |
| 21:25:31 | device locked (end of the missing period) |
| 21:27:04 | mobile data restored (after the window) |

(All times are UTC, a single world time standard, so records from different sources line up.
The case window written in `artifact_manifest.json` runs from 20:55:00 to 21:25:00. The unlock
and lock records sit right at its edges.)

The call and the WhatsApp activity fall inside the window. But the network record shows the phone
was **offline** from 21:10:00 to 21:26:59, so the image attachment at 21:13 cannot be called a
confirmed delivery. Saying "the image was delivered" would be an **overclaim**: a statement the
evidence does not support. Catching that is the whole point of replanning.

## Planning versus ReAct

| | **ReAct** (Lab 03) | **Planning** (this lab) |
|---|---|---|
| Starts with | the next single step | an ordered plan for the whole question |
| Good for | one check at a time | several connected steps that depend on each other |
| Adapts by | choosing the next action | revising the rest of the plan |

Think of organizing an event: *estimate guests -> pick a date -> book a room -> order food*. If the
room is taken, a good planner reorders the remaining steps instead of charging ahead. In this case,
the "room is taken" moment is the network record showing the phone was offline.

One more word you will see: a **replanning trigger** is a new fact that would force you to change
the plan. A good plan names its triggers up front.

## Part A: Walk the Plan-Replan Loop by Hand

In this part you drive the workflow yourself. You build a first plan from **part** of the
evidence, then the held-back record arrives and you ask for a revised plan, then a final report.

- **Step A.1:** package the evidence into a partial bundle and a held-back record.
- **Step A.2:** ask the AI for a first plan, using only the partial bundle.
- **Step A.3:** show the AI the network record and ask it to revise the plan.
- **Step A.4:** ask the AI for the final, evidence-based timeline.

Each step is one separate AI request. The AI does not remember earlier requests by itself, so each
step pastes in what it needs from the step before (for example, the plan from Step A.2).

### Step A.1: Package the evidence into bundles

**Purpose:** set up the twist. The first plan must be made **without** the network record, just
as a real examiner sometimes plans before every artifact has been found. The network record is
held back and arrives later.

**Input:** the six case files in the `data/` folder from Step 0.3.

**How it works:** the cell builds three **evidence bundles** (blocks of text that hold the contents
of several evidence files, pasted together so the AI can read them in one go):

- `partial_artifact_bundle`: the manifest, unlock events, call log, WhatsApp events, and chain of
  custody. It leaves out `network_status.csv` on purpose.
- `new_observation`: only `network_status.csv`, the late-arriving record.
- `full_artifact_bundle`: the partial bundle and the new observation joined together.

It reads the manifest in full and the first 8 lines of each other file. Every file in this case is
shorter than that, so in practice each whole file goes in.

**Output:** the line `Partial bundle omits network_status.csv; new_observation supplies it later.`
The three bundles are used next: the partial bundle in Steps A.2 and B.1, the new observation in
Steps A.3 and B.1, and the full bundle in Steps A.4 and B.2.

In [ ]:
# Package the evidence: a partial bundle (no network record), the held-back record, and everything.
from pathlib import Path
from IPython.display import Markdown, display  # show nicely formatted text (headings, bold) in the notebook

data_dir = Path('data')  # the folder of case files written in Step 0.3

# Helper: return the first n_lines lines of a file (8 unless you ask for more), so the AI gets
# a short, readable sample. Every file in this case is shorter than 8 lines.
def excerpt(filename, n_lines=8):
    lines = (data_dir / filename).read_text().strip().splitlines()
    return '\n'.join(lines[:n_lines])

# The partial bundle: each file's name followed by its contents. It leaves out network_status.csv
# on purpose, so the first plan is made without knowing the phone went offline.
partial_artifact_bundle = f'''
artifact_manifest.json
{(data_dir / 'artifact_manifest.json').read_text()}

unlock_events.csv
{excerpt('unlock_events.csv')}

call_log.csv
{excerpt('call_log.csv')}

whatsapp_events.csv
{excerpt('whatsapp_events.csv')}

chain_of_custody.csv
{excerpt('chain_of_custody.csv')}
'''

# The late-arriving record that will force a replan in Step A.3.
new_observation = f'''
network_status.csv
{excerpt('network_status.csv')}
'''

# Everything together, for the final report in Step A.4 and the agent run in Step B.2.
full_artifact_bundle = partial_artifact_bundle + '\n\n' + new_observation
print('Partial bundle omits network_status.csv; new_observation supplies it later.')

### Step A.2: Ask for a first investigation plan

**Purpose:** get an ordered plan for the investigation before drawing any conclusions. Because
the network record is still hidden, a good plan should name **replanning triggers** instead of
pretending it already has everything.

**Input:** `partial_artifact_bundle` from Step A.1, and `client` and `MODEL` from Step 0.2.

**How it works:** the cell writes three pieces of text:

- `PLANNING_PROMPT`: the standing instructions. It tells the AI to act as a forensics planning
  assistant and to avoid unsupported conclusions.
- `case_question`: the investigation question.
- `initial_plan_request`: the request itself. It asks for four parts and pastes in the question
  and the partial bundle.

It sends the instructions and the request to the AI and shows the reply as formatted text.

**Output:** a heading **Initial Investigation Plan**, then the AI's plan: (1) the investigation
goal, (2) ordered steps, (3) the evidence each step needs, and (4) replanning triggers. The plan
is saved as `initial_plan` for Step A.3. `PLANNING_PROMPT` is reused in Steps A.3 and A.4, and
`case_question` in Steps A.3, B.1, and B.2.

In [ ]:
# Ask the AI for a first, ordered plan, using only the partial evidence.

# The standing instructions: the AI's role and ground rules for every request in Part A.
PLANNING_PROMPT = (
    'You are a digital forensics planning assistant. Build ordered investigation plans, update them when new observations expose missing dependencies, and avoid unsupported conclusions.')

# The investigation question, reused in every later step.
case_question = (
    'Reconstruct the communication timeline during the incident window and determine which supported events occurred inside it. How should the timeline change if a network-status record is added later?')

# The request: what to return, the question, and the partial evidence pasted in at the end.
initial_plan_request = (
    'Build an initial investigation plan for this case.\n'
    'Return: (1) investigation goal, (2) ordered steps, (3) evidence needed, (4) replanning triggers.\n\n'
    f'Task:\n{case_question}\n\nArtifacts:\n{partial_artifact_bundle}')

# Send the instructions ('system') and the request ('user') to the AI; keep the text of its reply.
initial_plan = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': PLANNING_PROMPT},
              {'role': 'user', 'content': initial_plan_request}],
).choices[0].message.content
# Show the plan with its headings and bold text formatted.
display(Markdown('### Initial Investigation Plan\n\n' + initial_plan))

### Step A.3: Revise the plan after the network record arrives

**Purpose:** the network record shows the phone was offline from 21:10:00 to 21:26:59. That changes
what the 21:13 WhatsApp attachment can support. You ask the AI to **revise** its plan, not to
start over, so useful steps are kept and only what the new evidence affects is changed.

**Input:** `initial_plan` from Step A.2, `new_observation` from Step A.1, and `PLANNING_PROMPT` and
`case_question` from Step A.2.

**How it works:** the cell writes a revision request that pastes in the original question, the
current plan, and the new network record, and asks for four parts. It sends the request with the
same standing instructions as before.

**Output:** a heading **Observation-Driven Replan**, then: (1) what changed, (2) the revised steps,
(3) updated evidence priorities, and (4) remaining replanning triggers. Look for the AI saying
that the offline period means the attachment is not a confirmed delivery. The plan is saved as
`revised_plan` for Step A.4.

In [ ]:
# Ask the AI to revise its plan now that the network record has arrived.

# The request pastes in the question, the current plan from Step A.2, and the new record.
revised_plan_request = (
    'Revise the current investigation plan using the new observation below.\n'
    'Return: (1) what changed, (2) revised ordered steps, (3) updated evidence priorities, (4) remaining replanning triggers.\n\n'
    f'Original task:\n{case_question}\n\nCurrent plan:\n{initial_plan}\n\n'
    f'New observation:\n{new_observation}')

# Same instructions as Step A.2, new request. Keep the text of the reply and show it.
revised_plan = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': PLANNING_PROMPT},
              {'role': 'user', 'content': revised_plan_request}],
).choices[0].message.content
display(Markdown('### Observation-Driven Replan\n\n' + revised_plan))

### Step A.4: Write the final, evidence-based timeline

**Purpose:** turn the revised plan and all of the evidence into a short timeline report. It must
keep activity inside the window separate from anything the offline gap leaves unconfirmed.

**Input:** `revised_plan` from Step A.3, `full_artifact_bundle` from Step A.1, and `PLANNING_PROMPT`
from Step A.2.

**How it works:** the cell writes a report request that pastes in the revised plan and the full
evidence, asks for four parts, and sends it with the same standing instructions.

**Output:** a heading **Reconstructed Timeline and Timing Conclusion**, then: (1) the timeline,
(2) the timing conclusion, (3) an evidence mapping (which file supports each event), and (4) what
remains uncertain. It is saved as `final_timeline`. Check it against the case table above, and
use it when you write your own answer below.

In [ ]:
# Ask the AI for the final timeline report, using the revised plan and all of the evidence.

# The request pastes in the revised plan from Step A.3 and the full bundle from Step A.1.
final_report_request = (
    'Using the revised plan and full artifact set, produce a short evidence-cited timeline conclusion.\n'
    'Return: (1) reconstructed timeline, (2) timing conclusion, (3) evidence mapping, (4) what remains uncertain.\n\n'
    f'Revised plan:\n{revised_plan}\n\nArtifacts:\n{full_artifact_bundle}')

# Same instructions again, new request. Keep the text of the reply and show it.
final_timeline = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'system', 'content': PLANNING_PROMPT},
              {'role': 'user', 'content': final_report_request}],
).choices[0].message.content
display(Markdown('### Reconstructed Timeline and Timing Conclusion\n\n' + final_timeline))

## Part B: Run the Same Workflow with `PlanningAgent`

Now the packaged agent from Step 0.4 does the work. It wraps the same moves you made by hand:
build a plan, and revise it with a new observation. Its `run` method adds one more move: the AI
**reviews its own plan** and improves it until the review is satisfied. Your job is to compare
its work with your manual run.

- **Step B.1:** build and revise a plan through the agent.
- **Step B.2:** run the full plan, review, improve workflow.

### Step B.1: Build and revise a plan through the agent

**Purpose:** repeat Steps A.2 and A.3 with the packaged agent, and check that its revision is
driven by the new evidence, not by starting over.

**Input:** `case_question` from Step A.2, `partial_artifact_bundle` and `new_observation` from
Step A.1, `client` and `MODEL` from Step 0.2, and `PlanningAgent` from Step 0.4.

**How it works:** the cell creates the agent and joins the question and the partial evidence into
one task, `agent_task`. Then it makes two AI requests:

- `build_initial_plan` asks for a first plan in four parts, like Step A.2.
- `revise_plan` sends the task, that first plan, and the network record, and asks for a revised
  plan in four parts, like Step A.3.

The agent uses its own built-in instructions (`PLANNING_SYSTEM_PROMPT` from Step 0.4), a longer
version of the ones you wrote in Step A.2.

**Output:** two headings, **PlanningAgent Initial Plan** and **PlanningAgent Revised Plan**, each
followed by the AI's plan. The agent itself, `planning_agent`, is reused in Step B.2.

In [ ]:
# Use the packaged agent to build a first plan and then revise it, like Steps A.2 and A.3.
planning_agent = PlanningAgent(client=client, model=MODEL)

# One task text: the question plus the partial evidence (still no network record).
agent_task = f'{case_question}\n\nArtifacts:\n{partial_artifact_bundle}'
packaged_initial_plan = planning_agent.build_initial_plan(agent_task)
display(Markdown('### PlanningAgent Initial Plan\n\n' + packaged_initial_plan))

# Now hand the agent the late-arriving network record and ask for a revised plan.
packaged_revised_plan = planning_agent.revise_plan(
    user_msg=agent_task, current_plan=packaged_initial_plan, observations=new_observation)
display(Markdown('### PlanningAgent Revised Plan\n\n' + packaged_revised_plan))

### Step B.2: Run the full self-reviewing workflow

**Purpose:** see an agent check its own work. A plan that has been reviewed and fixed is usually
better than a first draft, but it is still a draft for you to judge.

**Input:** `planning_agent` from Step B.1, `case_question` from Step A.2, and
`full_artifact_bundle` from Step A.1 (this time the agent sees all of the evidence from the start).

**How it works:** `run` does three things:

1. It writes a first plan.
2. It asks the AI to review that plan. If the plan is well ordered, focused on evidence, and names
   clear replanning triggers, the AI should reply exactly `[[OK]]`. Otherwise it names the biggest
   gap.
3. If the review found a gap, it asks the AI to fix the plan using the review notes, then reviews
   again.

`max_replans=2` caps this at two reviews, because every review and fix is another AI request and
the free plan has limits. Unlike Part A, the agent keeps a running memory of this review
conversation, so each review builds on what came before. The instructions and the task always
stay in that memory; the oldest review messages drop out if it gets too long.

**Output:** possibly the line `Plan accepted by self-review. Stopping.`, then a heading
**PlanningAgent Final Plan** and the plan that came out of the review. This is an investigation
*plan*, not a finished timeline. Compare it with your Part A results when you answer the questions
below.

In [ ]:
# Let the agent plan, review its own plan, and improve it, using all of the evidence.
# max_replans=2: at most two review rounds, to save your free AI allowance.
packaged_final = planning_agent.run(
    user_msg=f'{case_question}\n\nArtifacts:\n{full_artifact_bundle}', max_replans=2)
display(Markdown('### PlanningAgent Final Plan\n\n' + packaged_final))

## Your Timeline and Decision

**Purpose:** write your own answer, as the examiner. The AI's plans and timeline above are drafts
to check, not the final word.

**Input:** the case records (the table in "The Case in One Minute" and the files in `data/`), and
the plans and timeline from Parts A and B.

**How it works:** the cell below holds your answer in four labelled parts. Replace each `'...'`
with your own words, then run the cell.

1. **Reconstructed timeline:** the key events, in order, with UTC times.
2. **Timing conclusion:** which events are inside the 20:55 to 21:25 window.
3. **Evidence mapping:** which artifact file supports each event.
4. **Remaining uncertainty:** what the offline gap (21:10:00 to 21:26:59) leaves unconfirmed,
   including the WhatsApp image attachment.

**Output:** your four parts printed, one per line. They are saved with the notebook when you submit.

In [ ]:
# Your answer. Replace each '...' with your own words. Add more items to the lists if you need them.
answer = {
    'timeline': ['... time + event, in order ...'],
    'in_window': ['... which events fall inside the missing period ...'],
    'evidence_mapping': {'... event ...': '... the artifact file that shows it ...'},
    'uncertainty': '... what the evidence does NOT confirm, and why ...',
}
# Print each part of your answer on its own line, so you can read it back.
for k, v in answer.items():
    print(k, '->', v)

### Compare the two runs

Look back at your hand-driven run (Part A) and the agent's run (Part B), then answer:

1. Did your manual replan and `PlanningAgent`'s replan reach the same careful conclusion?
2. Did either version keep the WhatsApp attachment as an **attempt** rather than a delivery?
3. Which single piece of evidence did the most to change the plan, and why?

## Submit

Save the notebook with Part A, Part B, and your answer filled in.

- In Colab: **File -> Download -> Download .ipynb**, or **File -> Save a copy in Drive**.

---

*Case data and the lab concept are adapted from*
*[frankwxu/agentic-AI4-forensics](https://github.com/frankwxu/agentic-AI4-forensics). The planning
framework is adapted from [The Neural Maze's agentic-patterns-course](https://github.com/neural-maze/agentic-patterns-course) (MIT License).*